# Week 2 — Task 2.1: Dataset Merging

Scope: validate keys, perform two left joins, save and verify one merged dataset. No preprocessing, filling, filtering, feature engineering, or modelling is performed.


## 1. Paths and source metadata


In [ ]:
from pathlib import Path
import json
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw'
OUTPUT = ROOT / 'data' / 'processed' / 'merged_energy_data.parquet'
STATS = ROOT / 'reports' / 'week2_merge_statistics.json'


In [ ]:
building = pd.read_csv(RAW / 'building_metadata.csv')
weather = pd.read_csv(RAW / 'weather_train.csv', parse_dates=['timestamp'])
print('Building metadata:', building.shape)
print('Weather:', weather.shape)


## 2. Validate right-side merge keys

Both right-side keys must be non-null and unique for safe many-to-one joins.


In [ ]:
assert building['building_id'].notna().all()
assert building['building_id'].is_unique
assert weather[['site_id', 'timestamp']].notna().all(axis=None)
assert not weather.duplicated(['site_id', 'timestamp']).any()
print('Right-side merge keys are valid.')


## 3. Run the memory-safe merge

The reusable project script reads training data in chunks, enforces `many_to_one` validation, and writes compressed Parquet. Running this cell recreates the output.


In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from scripts.week2_merge import merge_datasets
merge_statistics = merge_datasets()
merge_statistics['integrity']


## 4. Inspect merge statistics


In [ ]:
statistics = json.loads(STATS.read_text(encoding='utf-8'))
display(pd.DataFrame([statistics['source_rows']]))
display(pd.DataFrame([statistics['unmatched']]))
display(pd.DataFrame([statistics['success_rates']]))
display(pd.DataFrame([statistics['key_validation']]))


## 5. Independently verify the saved artifact


In [ ]:
parquet = pq.ParquetFile(OUTPUT)
assert parquet.metadata.num_rows == statistics['source_rows']['train']
assert parquet.schema_arrow.names == statistics['output']['column_names']
assert 'meter_reading' in parquet.schema_arrow.names
print(f"Verified {parquet.metadata.num_rows:,} rows and {len(parquet.schema_arrow.names)} columns.")
parquet.read_row_group(0).to_pandas().head()  # Memory-safe preview.


## 6. Task 2.1 conclusion

The merge is complete and integrity checks passed. Unmatched weather observations remain null. Stop here: preprocessing and all subsequent Week 2 tasks are intentionally outside this notebook.
